Основные изменения:
1. Изменена логика мэтчинга эмбеддингов и датасета
Убрана сложная система маппинга по ID
Эмбеддинги загружаются простым способом: индекс в массиве соответствует порядковому индексу в датасете
Созданы объединённые датасеты: df_train_with_embeddings, df_val_with_embeddings, df_test_with_embeddings
2. Добавлена проверка эмбеддингов
Функция check_embeddings_semantic_neighbors() загружает случайные примеры и находит их ближайших соседей
Выводит текст запроса, его метку и ближайших соседей с их сходством и метками
3. Удалена стратегия A, оставлена только B
Стратегия переименована в sample_semantic_balanced_top_n (семантический Top-N с балансировкой)
Название варианта: prompt_d_semantic_balanced_{dict}_{t}exmpls
4. Тестирование на тестовом датасете
input_jsons теперь формируется из df_test_with_embeddings
Метрики вычисляются на df_test
5. Изменена логика логирования
Логи содержат: id примера, количество few-shot примеров, id этих примеров
Затраты токенов и времени
Сам промпт не сохраняется в логи

In [18]:
from __future__ import annotations

import math
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path
import time
import csv
import re

import asyncio
import nest_asyncio
import random
from dataclasses import dataclass
from typing import Any

from openai import AsyncOpenAI, APIConnectionError, APIStatusError, RateLimitError
from tqdm.asyncio import tqdm

In [19]:
import logging
import sys
from io import StringIO
from datetime import datetime

# ─────────────────────────────────────────────
# Настройка директории логов и RUN_ID
# ─────────────────────────────────────────────

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        raise FileNotFoundError(f"Drive примонтирован, но {drive_root} недоступен")
    LOG_DIR = drive_root / "logs"
    USE_DRIVE = True
    print(f"✅ Google Drive подключён.")
except Exception as e:
    LOG_DIR = Path("logs")
    USE_DRIVE = False
    print(f"⚠ Drive недоступен ({e}), логи сохраняются локально в ./logs/")

LOG_DIR.mkdir(parents=True, exist_ok=True)
assert LOG_DIR.exists(), f"Не удалось создать директорию логов: {LOG_DIR}"

print(f"RUN_ID   : {RUN_ID}")
print(f"LOG_DIR  : {LOG_DIR}")
print(f"Папка существует: {LOG_DIR.exists()}")
print(f"USE_DRIVE: {USE_DRIVE}")

load_dotenv(".env")

BASE_URL        = os.getenv("BASE_URL")
API_KEY         = os.getenv("API_KEY")
MODEL_NAME      = "YandexGPT-5-Lite-8B-instruct"
MAX_CONCURRENCY = 256
TEMPERATURE     = 0
MAX_TEXT_LEN    = 1500

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive подключён.
RUN_ID   : 20260609_004201
LOG_DIR  : /content/drive/MyDrive/logs
Папка существует: True
USE_DRIVE: True


In [20]:
# ─────────────────────────────────────────────
# НАСТРОЙКА ПУТИ ЭМБЕДДИНГОВ
# ─────────────────────────────────────────────

EMBEDDINGS_DIR = Path(".")

In [22]:
# ─────────────────────────────────────────────
# ЗАГРУЗКА СЛОВАРЯ
# ─────────────────────────────────────────────

def load_drug_terms(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[tuple[str, str | None]] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            term = (row.get("normalized_term") or "").strip()
            cat_raw = (row.get("category") or "").strip()
            if not term:
                continue
            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue
            category: str | None = cat_raw if cat_raw else None
            key = (term, category)
            if key in seen:
                continue
            seen.add(key)
            items.append({"term": term, "category": category})

    items.sort(key=lambda x: (x["term"], x["category"] or ""))
    return json.dumps(items, ensure_ascii=False)


def load_drug_terms_short(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[str] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cat_raw = (row.get("category") or "").strip()
            if cat_raw != "drugs":
                continue
            term = (row.get("normalized_term") or "").strip()
            if not term:
                continue
            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue
            if term in seen:
                continue
            seen.add(term)
            items.append({"term": term, "category": "drugs"})

    items.sort(key=lambda x: x["term"])
    return json.dumps(items, ensure_ascii=False)


DRUG_TERMS       = load_drug_terms("illegal_terms_dictionary_edit.csv")
DRUG_TERMS_SHORT = load_drug_terms_short("illegal_terms_dictionary_edit.csv")

print(f"Загружено терминов в DRUG_TERMS: {len(json.loads(DRUG_TERMS))}")
print(f"Терминов в DRUG_TERMS_SHORT: {len(json.loads(DRUG_TERMS_SHORT))}")

Загружено терминов в DRUG_TERMS: 800
Терминов в DRUG_TERMS_SHORT: 104


In [23]:
# ─────────────────────────────────────────────
# ЗАГРУЗКА ДАТАСЕТОВ
# ─────────────────────────────────────────────

df_val   = pd.read_parquet("val.parquet").reset_index(drop=True)
df_test  = pd.read_parquet("test.parquet").reset_index(drop=True)
df_train = pd.read_parquet("train.parquet").reset_index(drop=True)


def row_to_input_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":   f"{row['session_id']}___{idx}",
        "text": text[:MAX_TEXT_LEN],
    }


input_jsons = [row_to_input_json(row, idx) for idx, row in df_val.iterrows()]

print(f"\nВсего записей в val (подбор гиперпараметров): {len(input_jsons)}")
print(f"Уникальных session_id: {len({i['id'] for i in input_jsons})}")
print("\nПример:")
print(json.dumps(input_jsons[0], ensure_ascii=False, indent=2))

assert len({i["id"] for i in input_jsons}) == len(input_jsons), (
    "session_id не уникален в val.parquet!"
)



Всего записей в val (подбор гиперпараметров): 336
Уникальных session_id: 336

Пример:
{
  "id": "telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-5452102867.2de8f5e4-9fcc-c2b2-2acc-63fd3ca1803b___0",
  "text": "Вопрос: /newNode_7;Альфа-ПВП VHQ+ТОП 0.5гр.280грн\nОтвет: Избран продукт: Альфа-ПВП VHQ+ ТОП Прозрачные крисы 0.5 гр.\nКоротко о товаре: Всеми любимая и знакомая Альфа пвп. Очень мощные, прозрачные криссталы, высокого качества!!!\nЦена: 280 грн.\nВыберите подходящий район:"
}


In [24]:
# ─────────────────────────────────────────────
# ПОДГОТОВКА ДАННЫХ TRAIN ДЛЯ FEW-SHOT
# ─────────────────────────────────────────────

SEED_VALUE = 42
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)

T_GRID = [2, 4, 8, 16, 32]


def row_to_train_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":    f"{row['session_id']}___{idx}",
        "text":  text[:MAX_TEXT_LEN],
        "label": int(row["from_illegal_account"]),
    }


examples_jsons = [row_to_train_json(row, idx) for idx, row in df_train.iterrows()]

examples_id    = [item["id"]    for item in examples_jsons]
examples_text  = [item["text"]  for item in examples_jsons]
examples_label = [item["label"] for item in examples_jsons]

print(f"\nВсего записей в train : {len(examples_id)}")
print(f"Уникальных session_id : {len(set(examples_id))}")
print(f"Legal   (0)           : {examples_label.count(0)}")
print(f"Illegal (1)           : {examples_label.count(1)}")

assert len(set(examples_id)) == len(examples_id), (
    "session_id не уникален в train.parquet!"
)

legal_indices   = [i for i, lbl in enumerate(examples_label) if lbl == 0]
illegal_indices = [i for i, lbl in enumerate(examples_label) if lbl == 1]




Всего записей в train : 748
Уникальных session_id : 748
Legal   (0)           : 182
Illegal (1)           : 566


In [25]:
# ─────────────────────────────────────────────
# ЗАГРУЗКА ЭМБЕДДИНГОВ (ИЗМЕНЁННАЯ ЛОГИКА)
# ─────────────────────────────────────────────

def load_embeddings_by_index(split_name: str, emb_dir: Path = EMBEDDINGS_DIR) -> np.ndarray:
    """
    Загружает эмбеддинги для сплита.
    Индекс в массиве эмбеддингов соответствует порядковому индексу в датасете.

    Args:
        split_name: "train", "val" или "test"
        emb_dir: директория с .npy файлами

    Returns:
        np.ndarray (N, D) float32 - эмбеддинги
    """
    emb_path = emb_dir / f"{split_name}_embeddings.npy"

    if not emb_path.exists():
        raise FileNotFoundError(f"Файл эмбеддингов не найден: {emb_path}")

    embeddings = np.load(emb_path, allow_pickle=True).astype(np.float32)

    # Проверяем нормализацию
    n_check = min(100, len(embeddings))
    sample_norms = np.linalg.norm(embeddings[:n_check], axis=1)
    already_normalized = bool(np.allclose(sample_norms, 1.0, atol=1e-3))

    if not already_normalized:
        print(f"  [{split_name}] ⚠ Эмбеддинги не нормализованы, нормализуем")
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        embeddings = (embeddings / norms).astype(np.float32)

    print(f"  [{split_name}] shape={embeddings.shape} | normalized={already_normalized}")

    return embeddings


print("\nЗагружаем эмбеддинги...")
train_embeddings = load_embeddings_by_index("train")
val_embeddings   = load_embeddings_by_index("val")
test_embeddings  = load_embeddings_by_index("test")



Загружаем эмбеддинги...
  [train] ⚠ Эмбеддинги не нормализованы, нормализуем
  [train] shape=(748, 4096) | normalized=False
  [val] ⚠ Эмбеддинги не нормализованы, нормализуем
  [val] shape=(336, 4096) | normalized=False
  [test] ⚠ Эмбеддинги не нормализованы, нормализуем
  [test] shape=(305, 4096) | normalized=False


In [26]:
# ─────────────────────────────────────────────
# ОБЪЕДИНЕНИЕ ДАТАСЕТОВ С ЭМБЕДДИНГАМИ
# ─────────────────────────────────────────────

# Проверка соответствия размеров
assert len(df_train) == len(train_embeddings), (
    f"Размер train датасета ({len(df_train)}) не совпадает с эмбеддингами ({len(train_embeddings)})"
)
assert len(df_val) == len(val_embeddings), (
    f"Размер val датасета ({len(df_val)}) не совпадает с эмбеддингами ({len(val_embeddings)})"
)
assert len(df_test) == len(test_embeddings), (
    f"Размер test датасета ({len(df_test)}) не совпадает с эмбеддингами ({len(test_embeddings)})"
)

# Создаём объединённые датасеты
df_train_with_embeddings = df_train.copy()
df_train_with_embeddings['embedding'] = list(train_embeddings)

df_val_with_embeddings = df_val.copy()
df_val_with_embeddings['embedding'] = list(val_embeddings)

df_test_with_embeddings = df_test.copy()
df_test_with_embeddings['embedding'] = list(test_embeddings)

print(f"\n✅ Объединение выполнено:")
print(f"  df_train_with_embeddings: {len(df_train_with_embeddings)} записей")
print(f"  df_val_with_embeddings:   {len(df_val_with_embeddings)} записей")
print(f"  df_test_with_embeddings:  {len(df_test_with_embeddings)} записей")


✅ Объединение выполнено:
  df_train_with_embeddings: 748 записей
  df_val_with_embeddings:   336 записей
  df_test_with_embeddings:  305 записей


In [27]:
# ─────────────────────────────────────────────
# ПРОВЕРКА ЭМБЕДДИНГОВ - ПОИСК БЛИЖАЙШИХ СОСЕДЕЙ
# ─────────────────────────────────────────────

def check_embeddings_semantic_neighbors(
    df_with_emb: pd.DataFrame,
    n_random: int = 3,
    n_neighbors: int = 5
):
    """
    Проверка качества эмбеддингов: для случайных примеров находим ближайших соседей
    и проверяем их семантическую близость.

    Args:
        df_with_emb: датасет с колонкой 'embedding'
        n_random: количество случайных примеров для проверки
        n_neighbors: количество ближайших соседей для каждого примера
    """
    print(f"\n{'='*60}")
    print(f"ПРОВЕРКА ЭМБЕДДИНГОВ - ПОИСК БЛИЖАЙШИХ СОСЕДЕЙ")
    print(f"{'='*60}")

    # Собираем матрицу эмбеддингов
    emb_matrix = np.stack(df_with_emb['embedding'].values)

    # Выбираем случайные индексы
    random_indices = random.sample(range(len(df_with_emb)), min(n_random, len(df_with_emb)))

    for idx in random_indices:
        query_emb = emb_matrix[idx]

        # Вычисляем косинусное сходство со всеми
        similarities = emb_matrix @ query_emb  # dot product для нормализованных

        # Находим топ-N ближайших (исключая сам пример)
        # Сортируем по убыванию сходства
        nearest_indices = np.argsort(similarities)[::-1]
        # Исключаем сам запрос
        nearest_indices = [i for i in nearest_indices if i != idx][:n_neighbors]

        print(f"\n{'─'*60}")
        print(f"Запрос (индекс {idx}):")
        print(f"  Текст: {df_with_emb.iloc[idx]['question'][:200]}...")
        print(f"  Метка: {df_with_emb.iloc[idx]['from_illegal_account']}")

        print(f"\n  Ближайшие соседи:")
        for rank, neighbor_idx in enumerate(nearest_indices, 1):
            sim = similarities[neighbor_idx]
            text = df_with_emb.iloc[neighbor_idx]['question'][:200]
            label = df_with_emb.iloc[neighbor_idx]['from_illegal_account']
            print(f"  {rank}. [idx={neighbor_idx}] sim={sim:.4f} | label={label}")
            print(f"     {text}...")

    print(f"\n{'='*60}")
    print("ПРОВЕРКА ЗАВЕРШЕНА")
    print(f"{'='*60}")


# Запускаем проверку на train датасете
check_embeddings_semantic_neighbors(df_train_with_embeddings, n_random=3, n_neighbors=5)



ПРОВЕРКА ЭМБЕДДИНГОВ - ПОИСК БЛИЖАЙШИХ СОСЕДЕЙ

────────────────────────────────────────────────────────────
Запрос (индекс 654):
  Текст: добрый день! подскажите, можно ли оформить студенческую визу на ребенка 16 лет? приглашение из школы будет.
__
from user: @irakli_77...
  Метка: 0

  Ближайшие соседи:
  1. [idx=384] sim=0.4899 | label=0
     а могу в лиссабон с внж испании обратиться?
__
from user: @narkisshir...
  2. [idx=168] sim=0.4694 | label=0
     📌 требуется фрилансер веб дизайнер, писать в личные сообщения. рассматриваю лица достигшие старше 18 ле
__
from user: @isa_2008...
  3. [idx=570] sim=0.4638 | label=0
     кто занимается инфографикой? отпишите мне, работа есть) (только от 18 лет, кто не достиг возраста, прошу не беспокоить)
__
from user: @samuil_43...
  4. [idx=104] sim=0.4474 | label=0
     инфографику кто может сделать? на женские куртки./ рассматриваю лица достигшие старше 18 лет
__
from user: @strelkovav...
  5. [idx=655] sim=0.4445 | label=0
     ищу дизайнера

In [28]:
# ─────────────────────────────────────────────
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ВЫВОДА
# ─────────────────────────────────────────────

def print_samples(indices: list, title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for idx in sorted(indices):
        label_str = "illegal" if examples_label[idx] == 1 else "legal"
        print(f"\n[{label_str}]")
        print(examples_text[idx])
        print("-" * 40)


In [29]:
# ─────────────────────────────────────────────
# СТРАТЕГИИ ОТБОРА ПРИМЕРОВ
# ─────────────────────────────────────────────

def sample_random(t: int) -> list:
    """Случайная выборка t примеров."""
    return random.sample(range(len(examples_id)), t)


def sample_balanced(t: int) -> list:
    """Сбалансированная выборка t примеров (1:1 legal:illegal)."""
    n_illegal = t // 2
    n_legal   = t // 2
    remainder = t % 2

    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)

    if remainder:
        extra_pool = (
            [i for i in legal_indices   if i not in sampled_legal] +
            [i for i in illegal_indices if i not in sampled_illegal]
        )
        sampled_illegal += random.sample(extra_pool, remainder)

    result = sampled_legal + sampled_illegal
    assert len(result) == t, f"balanced: ожидалось {t}, получено {len(result)}"
    return result

'''
def sample_2to1(t: int) -> list:
    """Выборка t примеров с соотношением 2:1 (legal:illegal)."""
    n_illegal = max(1, round(t / 3))
    n_legal   = t - n_illegal
    n_illegal = min(n_illegal, len(illegal_indices))
    n_legal   = min(n_legal,   len(legal_indices))
    actual_t  = n_legal + n_illegal

    result = (
        random.sample(legal_indices,   n_legal) +
        random.sample(illegal_indices, n_illegal)
    )
    assert len(result) == actual_t
    return result
'''

'\ndef sample_2to1(t: int) -> list:\n    """Выборка t примеров с соотношением 2:1 (legal:illegal)."""\n    n_illegal = max(1, round(t / 3))\n    n_legal   = t - n_illegal\n    n_illegal = min(n_illegal, len(illegal_indices))\n    n_legal   = min(n_legal,   len(legal_indices))\n    actual_t  = n_legal + n_illegal\n\n    result = (\n        random.sample(legal_indices,   n_legal) +\n        random.sample(illegal_indices, n_illegal)\n    )\n    assert len(result) == actual_t\n    return result\n'

In [30]:
# ─────────────────────────────────────────────
# СТРАТЕГИЯ: Семантический Top-N с балансировкой по классам
# ─────────────────────────────────────────────

def sample_semantic_balanced_top_n(
    query_emb: np.ndarray,
    t: int,
    order: str = "closest_last",
) -> list[int]:
    """
    Стратегия семантического отбора с балансировкой по классам.

    Выбирает N/2 наиболее похожих legal и N/2 наиболее похожих illegal примеров.

    Шаги:
    1) Разделить примеры из обучающей выборки по классам
    2) Вычислить косинусное сходство между эмбеддингом тестового документа и каждым эмбеддингом из обучающей выборки
    3) Выбрать:
       - N/2 наиболее похожих legal примеров
       - N/2 наиболее похожих illegal примеров
    4) Упорядочить примеры в промпте (closest-last: менее похожие первыми, наиболее похожие ближе к тестовому документу)

    Args:
        query_emb: нормализованный эмбеддинг запроса, shape (D,)
        t:         общее количество примеров
        order:     "closest_last" — менее похожие первыми, наиболее похожие последними

    Returns:
        Список глобальных индексов в нужном порядке для промпта.
    """
    n_illegal = t // 2
    n_legal   = t - n_illegal  # при нечётном t: legal на 1 больше

    # Строим матрицы эмбеддингов по классам
    train_emb_matrix = np.stack([emb for emb in df_train_with_embeddings['embedding']])

    legal_mask = df_train_with_embeddings['from_illegal_account'] == 0
    illegal_mask = df_train_with_embeddings['from_illegal_account'] == 1

    legal_embeddings = train_emb_matrix[legal_mask]
    illegal_embeddings = train_emb_matrix[illegal_mask]

    legal_indices_global = df_train_with_embeddings[legal_mask].index.tolist()
    illegal_indices_global = df_train_with_embeddings[illegal_mask].index.tolist()

    # Вычисляем косинусное сходство
    sim_legal = legal_embeddings @ query_emb
    sim_illegal = illegal_embeddings @ query_emb

    def _top_k(sim_vec: np.ndarray, global_indices: list, k: int) -> list[tuple[int, float]]:
        """Возвращает топ-k индексов с их сходствами."""
        k = min(k, len(sim_vec))
        if k == 0:
            return []
        if k < len(sim_vec):
            top_mi = np.argpartition(sim_vec, -k)[-k:]
            top_mi = top_mi[np.argsort(sim_vec[top_mi])[::-1]]
        else:
            top_mi = np.argsort(sim_vec)[::-1]
        return [(global_indices[mi], float(sim_vec[mi])) for mi in top_mi]

    # Выбираем топ-k по каждому классу
    top_legal = _top_k(sim_legal, legal_indices_global, n_legal)
    top_illegal = _top_k(sim_illegal, illegal_indices_global, n_illegal)

    # Объединяем и сортируем по возрастанию сходства (closest-last)
    combined = top_legal + top_illegal
    combined_asc = sorted(combined, key=lambda x: x[1])  # от меньшего сходства к большему

    return [idx for idx, _ in combined_asc]


In [31]:
# ─────────────────────────────────────────────
# SemanticSampler — callable-адаптер для семантических стратегий
# ─────────────────────────────────────────────

class SemanticSampler:
    """
    Callable-объект для семантической выборки примеров.

    Использование:
        sampler = SemanticSampler(query_emb, order="closest_last")
        indices = sampler(t=8)  # → list[int]
    """

    def __init__(
        self,
        query_emb: np.ndarray,
        order: str = "closest_last",
    ):
        self.query_emb = query_emb
        self.order = order

    def __call__(self, t: int) -> list[int]:
        return sample_semantic_balanced_top_n(self.query_emb, t, self.order)

In [32]:
# ─────────────────────────────────────────────
# ПРОМПТЫ
# ─────────────────────────────────────────────

SYSTEM_PROMPT = "Ты - помощник по классификации текста для задачи модерации на предмет упоминания наркотиков."

prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Прямое упоминание сущностей, связанных с наркотиками и наркоторговлей:
   - обменник, клад, кладмен, фасовка, закладка и т.д.

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова, особенно в названиях каналов и ботов через @:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - добавление лишних символов: DeaIler
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Подозрительные аббревиатуры и одиночные буквы латиницей в качестве вопроса:
   - Bbgg, Sh, I, CV GK j

6. Упоминания криптокошельков и криптовалют

7. Фразы с двойным дном и иносказания:
   - Главное не забывать: счастье — это когда ты нашёл, а тебя нет!

8. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

9. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ПРИМЕРЫ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ:
{FEW_SHOT_EXAMPLES}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

In [33]:
KNOWN_PLACEHOLDERS = {"FEW_SHOT_EXAMPLES", "DRUG_TERMS", "INPUT_JSON"}


def escape_unknown_placeholders(text: str, known: set) -> str:
    def replacer(m):
        key = m.group(1)
        return f"{{{{{key}}}}}" if key not in known else m.group(0)
    return re.sub(r'\{(\w+)\}', replacer, text)


prompt_d = escape_unknown_placeholders(prompt_d, KNOWN_PLACEHOLDERS)

remaining = re.findall(r'\{(\w+)\}', prompt_d)
print("Оставшиеся плейсхолдеры:", remaining)

Оставшиеся плейсхолдеры: ['DRUG_TERMS', 'FEW_SHOT_EXAMPLES', 'INPUT_JSON']


In [34]:
# ─────────────────────────────────────────────
# ПОСТРОИТЕЛЬ FEW-SHOT БЛОКА
# ─────────────────────────────────────────────

def build_few_shot_block(indices: list) -> str:
    """
    Формирует строку с примерами для вставки в промпт.
    """
    blocks = []
    for idx in indices:
        label     = bool(examples_label[idx] == 1)
        label_str = "ILLEGAL" if label else "LEGAL"

        example_input = json.dumps(
            {"text": examples_text[idx]},
            ensure_ascii=False,
            indent=2,
        )
        example_output = json.dumps(
            {"has_drug_mention": label},
            ensure_ascii=False,
            indent=2,
        )
        blocks.append(
            f"[{label_str}]\n"
            f"ПРИМЕР ВХОДА:\n{example_input}\n\n"
            f"ПРИМЕР ВЫХОДА:\n{example_output}"
        )
    return "\n\n" + ("\n\n" + "─" * 40 + "\n\n").join(blocks) + "\n"



In [35]:
# ─────────────────────────────────────────────
# ПОСТРОИТЕЛЬ СООБЩЕНИЙ
# ─────────────────────────────────────────────

def build_messages_d(
    item,
    sample_fn=None,
    use_dict: bool = True,
    t: int = 10,
):
    """
    Строит список сообщений для API.
    """
    if sample_fn is not None:
        indices        = sample_fn(t)
        few_shot_block = build_few_shot_block(indices)
    else:
        few_shot_block = "(примеры не используются)"

    drug_terms_value = DRUG_TERMS if use_dict else "(словарь не используется)"

    user_content = prompt_d.format(
        FEW_SHOT_EXAMPLES=few_shot_block,
        DRUG_TERMS=drug_terms_value,
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


def build_messages_semantic(
    item: dict,
    use_dict: bool,
    t: int,
    order: str = "closest_last",
) -> list[dict]:
    """
    Построитель сообщений для семантической стратегии.
    Использует эмбеддинги из df_test_with_embeddings.
    """
    # Ищем эмбеддинг для данного item в тестовом датасете
    item_id = item["id"]

    # Извлекаем session_id и row_idx из item_id
    parts = item_id.split("___")
    if len(parts) == 2:
        session_id, row_idx = parts
        row_idx = int(row_idx)
    else:
        # Fallback: ищем по полному id
        query_emb = None

    # Ищем эмбеддинг в df_test_with_embeddings по индексу
    try:
        query_emb = df_test_with_embeddings.iloc[row_idx]['embedding']
    except (IndexError, KeyError):
        query_emb = None

    if query_emb is None:
        # Fallback на случайную выборку если эмбеддинг не найден
        sampler = sample_random
    else:
        sampler = SemanticSampler(
            query_emb=query_emb,
            order=order,
        )

    return build_messages_d(item, sample_fn=sampler, use_dict=use_dict, t=t)


In [43]:
# ─────────────────────────────────────────────
# ГЕНЕРАЦИЯ PROMPT_VARIANTS
# ─────────────────────────────────────────────

PROMPT_VARIANTS: dict[str, callable] = {}

# Базовые конфигурации: (strategy_name, sample_fn, use_dict)
BASE_CONFIGS = [
    ("random",   sample_random,   False),
    ("balanced", sample_balanced, False),
    ("random",   sample_random,   True),
    ("balanced", sample_balanced, True),
]

# Базовые стратегии (без семантики) - перебираем все комбинации с T_GRID
for _strategy, _sample_fn, _use_dict in BASE_CONFIGS:
    for _t in T_GRID:
        _dict_suf = "with_dict" if _use_dict else "wo_dict"
        _name = f"prompt_d_{_strategy}_{_dict_suf}_{_t}exmpls"
        PROMPT_VARIANTS[_name] = (
            lambda x, fn=_sample_fn, ud=_use_dict, t_val=_t:
                build_messages_d(x, sample_fn=fn, use_dict=ud, t=t_val)
        )

# Семантические стратегии
SEMANTIC_STRATEGIES = [
    ("semantic_random",   sample_random,   "random"),
    ("semantic_balanced", sample_balanced, "balanced"),
]

for _strategy_name, _sample_fn, _base_name in SEMANTIC_STRATEGIES:
    for _use_dict in [False, True]:
        for _t in T_GRID:
            _dict_suf = "with_dict" if _use_dict else "wo_dict"
            _name = f"prompt_d_{_strategy_name}_{_dict_suf}_{_t}exmpls"

            if _base_name == "balanced":
                # Для balanced используем семантическую выборку с балансировкой
                PROMPT_VARIANTS[_name] = (
                    lambda x, ud=_use_dict, t_val=_t:
                        build_messages_semantic(
                            x, use_dict=ud, t=t_val, order="closest_last"
                        )
                )
            else:
                # Для random используем обычную случайную выборку
                PROMPT_VARIANTS[_name] = (
                    lambda x, fn=_sample_fn, ud=_use_dict, t_val=_t:
                        build_messages_d(x, sample_fn=fn, use_dict=ud, t=t_val)
                )

print(f"\nВсего вариантов промптов: {len(PROMPT_VARIANTS)}")
print("\nВарианты:")
for name in sorted(PROMPT_VARIANTS.keys()):
    print(f"  {name}")


Всего вариантов промптов: 40

Варианты:
  prompt_d_balanced_with_dict_16exmpls
  prompt_d_balanced_with_dict_2exmpls
  prompt_d_balanced_with_dict_32exmpls
  prompt_d_balanced_with_dict_4exmpls
  prompt_d_balanced_with_dict_8exmpls
  prompt_d_balanced_wo_dict_16exmpls
  prompt_d_balanced_wo_dict_2exmpls
  prompt_d_balanced_wo_dict_32exmpls
  prompt_d_balanced_wo_dict_4exmpls
  prompt_d_balanced_wo_dict_8exmpls
  prompt_d_random_with_dict_16exmpls
  prompt_d_random_with_dict_2exmpls
  prompt_d_random_with_dict_32exmpls
  prompt_d_random_with_dict_4exmpls
  prompt_d_random_with_dict_8exmpls
  prompt_d_random_wo_dict_16exmpls
  prompt_d_random_wo_dict_2exmpls
  prompt_d_random_wo_dict_32exmpls
  prompt_d_random_wo_dict_4exmpls
  prompt_d_random_wo_dict_8exmpls
  prompt_d_semantic_balanced_with_dict_16exmpls
  prompt_d_semantic_balanced_with_dict_2exmpls
  prompt_d_semantic_balanced_with_dict_32exmpls
  prompt_d_semantic_balanced_with_dict_4exmpls
  prompt_d_semantic_balanced_with_dict_8e

In [44]:
# ─────────────────────────────────────────────
# НАСТРОЙКА ЛОГИРОВАНИЯ (ИЗМЕНЁННАЯ)
# ─────────────────────────────────────────────

class ColaFormatter(logging.Formatter):
    COLORS = {
        logging.DEBUG:    "\033[37m",
        logging.INFO:     "\033[36m",
        logging.WARNING:  "\033[33m",
        logging.ERROR:    "\033[31m",
        logging.CRITICAL: "\033[35m",
    }
    RESET = "\033[0m"

    def format(self, record):
        color = self.COLORS.get(record.levelno, self.RESET)
        record.levelname = f"{color}{record.levelname:<8}{self.RESET}"
        return super().format(record)


def setup_logger(
    name: str,
    log_file: str,
    console_level: int = logging.INFO,
    file_level: int    = logging.DEBUG,
) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    if logger.handlers:
        logger.handlers.clear()

    file_fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_fmt = ColaFormatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    fh = logging.FileHandler(LOG_DIR / log_file, encoding="utf-8", mode="a")
    fh.setLevel(file_level)
    fh.setFormatter(file_fmt)
    logger.addHandler(fh)

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(console_level)
    ch.setFormatter(console_fmt)
    logger.addHandler(ch)

    buffer = StringIO()
    bh = logging.StreamHandler(buffer)
    bh.setLevel(logging.DEBUG)
    bh.setFormatter(file_fmt)
    logger.addHandler(bh)

    logger.buffer = buffer
    return logger


logger       = setup_logger("main",  f"run_{RUN_ID}.log",          console_level=logging.INFO)
api_logger   = setup_logger("api",   f"api_{RUN_ID}.log",          console_level=logging.WARNING)
parse_logger = setup_logger("parse", f"parse_errors_{RUN_ID}.log", console_level=logging.WARNING)

logger.info(f"Логирование настроено | RUN_ID={RUN_ID}")
logger.info(f"Логи сохраняются в: {LOG_DIR}")
logger.info(f"Google Drive: {'подключён' if USE_DRIVE else 'не используется'}")
logger.info(f"Всего вариантов промптов: {len(PROMPT_VARIANTS)}")


01:43:21 | INFO     | main | Логирование настроено | RUN_ID=20260609_004201


INFO    :main:Логирование настроено | RUN_ID=20260609_004201


01:43:21 | INFO     | main | Логи сохраняются в: /content/drive/MyDrive/logs


INFO    :main:Логи сохраняются в: /content/drive/MyDrive/logs


01:43:21 | INFO     | main | Google Drive: подключён


INFO    :main:Google Drive: подключён


01:43:21 | INFO     | main | Всего вариантов промптов: 40


INFO    :main:Всего вариантов промптов: 40


In [45]:
# ─────────────────────────────────────────────
# УТИЛИТЫ ЛОГОВ
# ─────────────────────────────────────────────

def show_logs(logger_name: str = "main", tail: int = 50):
    log = logging.getLogger(logger_name)
    if not hasattr(log, "buffer"):
        print("Буфер не найден")
        return
    lines = log.buffer.getvalue().splitlines()
    print(f"\n=== Последние {tail} строк лога [{logger_name}] ===")
    for line in lines[-tail:]:
        print(line)


def show_log_files():
    print(f"\n=== Файлы логов в {LOG_DIR} ===")
    files = sorted(LOG_DIR.glob("*.log"))
    if not files:
        print("  (пусто)")
        return
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<45} {size_kb:>8.1f} KB")


def download_logs():
    try:
        from google.colab import files
        log_files = sorted(LOG_DIR.glob("*.log"))
        if not log_files:
            print("Нет файлов для скачивания")
            return
        for f in log_files:
            print(f"Скачиваем: {f.name}")
            files.download(str(f))
    except ImportError:
        print("Функция доступна только в Google Colab")


def clear_log_buffer(logger_name: str = "main"):
    log = logging.getLogger(logger_name)
    if hasattr(log, "buffer"):
        log.buffer.truncate(0)
        log.buffer.seek(0)
        print(f"Буфер [{logger_name}] очищен")


In [46]:

# ─────────────────────────────────────────────
# АСИНХРОННЫЕ ЗАПРОСЫ
# ─────────────────────────────────────────────

async def send_one_request(
    client,
    model_name: str,
    messages: list,
    item_id: str = "unknown",
    few_shot_ids: list = None,
    n_examples: int = 0,
):
    """
    Отправляет один запрос к API.

    Args:
        few_shot_ids: список id few-shot примеров (для логирования)
        n_examples: количество few-shot примеров
    """
    api_logger.debug(
        f"[{item_id}] → Запрос | "
        f"prompt_len={sum(len(m['content']) for m in messages)}"
    )
    start = time.time()
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=messages,
            max_tokens=2048,
            temperature=0.0,
            seed=42,
        )
        elapsed           = time.time() - start
        content           = response.choices[0].message.content
        prompt_tokens     = response.usage.prompt_tokens
        completion_tokens = response.usage.completion_tokens
        total_tokens      = prompt_tokens + completion_tokens

        # Логируем информацию о запросе (без самого промпта)
        api_logger.info(
            f"[{item_id}] Запрос выполнен | "
            f"time={elapsed:.2f}s | "
            f"prompt_tokens={prompt_tokens} | "
            f"completion_tokens={completion_tokens} | "
            f"total_tokens={total_tokens} | "
            f"n_examples={n_examples} | "
            f"few_shot_ids={few_shot_ids[:5] if few_shot_ids else 'none'}..."
        )

        api_logger.debug(
            f"[{item_id}] ← Ответ | "
            f"preview={content[:60].replace(chr(10),' ')!r}"
        )

        return {
            "response":          content,
            "time":              elapsed,
            "prompt_tokens":     prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens":      total_tokens,
        }

    except RateLimitError as e:
        elapsed = time.time() - start
        api_logger.warning(f"[{item_id}] RateLimitError | time={elapsed:.2f}s | {e}")
        raise

    except APIConnectionError as e:
        elapsed = time.time() - start
        api_logger.error(f"[{item_id}] APIConnectionError | time={elapsed:.2f}s | {e}")
        raise

    except APIStatusError as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] APIStatusError | "
            f"time={elapsed:.2f}s | "
            f"status={e.status_code} | "
            f"body={str(e.body)[:200]}"
        )
        raise

    except Exception as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] UnexpectedError | "
            f"time={elapsed:.2f}s | "
            f"{type(e).__name__}: {e}"
        )
        raise


async def process_with_semaphore(
    client,
    model_name: str,
    messages: list,
    item_id: str = "unknown",
    few_shot_ids: list = None,
    n_examples: int = 0,
):
    async with semaphore:
        return await send_one_request(
            client, model_name, messages,
            item_id=item_id,
            few_shot_ids=few_shot_ids,
            n_examples=n_examples,
        )


def parse_response(raw: str, item_id: str):
    try:
        cleaned = re.sub(r"```(?:json)?|```", "", raw).strip()
        data    = json.loads(cleaned)
        result  = {
            "id":               str(data.get("id", item_id)),
            "has_drug_mention": bool(data.get("has_drug_mention", False)),
        }
        api_logger.debug(
            f"[{item_id}] Парсинг OK | "
            f"has_drug_mention={result['has_drug_mention']}"
        )
        return result

    except json.JSONDecodeError as e:
        parse_logger.error(
            f"[{item_id}] JSONDecodeError: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None

    except Exception as e:
        parse_logger.error(
            f"[{item_id}] ParseError: {type(e).__name__}: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None


In [47]:
# ─────────────────────────────────────────────
# ОЦЕНКА МЕТРИК
# ─────────────────────────────────────────────

def evaluate_results(
    results_file: str,
    df_truth: pd.DataFrame,
    label: str,
    truth_label_col: str = "message_label",
):
    if not Path(results_file).exists():
        print(f"[{label}] файл {results_file} не найден, пропускаю")
        return None

    with open(results_file) as f:
        preds = json.load(f)

    df_pred = pd.DataFrame(preds)
    if df_pred.empty:
        print(f"[{label}] файл пустой, пропускаю")
        return None

    df_pred = df_pred.drop_duplicates(subset="id", keep="last")
    df_pred["id"] = df_pred["id"].astype(str)

    df_truth_local = df_truth.copy()
    df_truth_local["composite_id"] = (
        df_truth_local["session_id"].astype(str) + "___" +
        df_truth_local.index.astype(str)
    )

    expected_ids = set(df_truth_local["composite_id"])
    actual_ids   = set(df_pred["id"])
    missing = expected_ids - actual_ids
    extra   = actual_ids   - expected_ids
    if missing or extra:
        print(
            f"[{label}] ВНИМАНИЕ: "
            f"пропущено id из truth: {len(missing)}, "
            f"лишних id в pred: {len(extra)}"
        )
        if extra:
            print(f"  пример лишних id: {list(extra)[:3]}")
        if missing:
            print(f"  пример пропущенных id: {list(missing)[:3]}")

    df_merged = df_truth_local.merge(
        df_pred, left_on="composite_id", right_on="id", how="inner"
    )

    if len(df_merged) == 0:
        print(f"[{label}] нет совпадений по id, пропускаю.")
        return None

    if len(df_merged) != len(df_truth_local):
        print(
            f"[{label}] предупреждение: "
            f"смержилось {len(df_merged)} из {len(df_truth_local)} строк"
        )

    y_true = (df_merged[truth_label_col] == "illegal").astype(int)
    y_pred = df_merged["has_drug_mention"].astype(int)

    metrics = {
        "version":   label,
        "n":         len(df_merged),
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1_binary": f1_score(y_true, y_pred, average="binary",  zero_division=0),
        "f1_macro":  f1_score(y_true, y_pred, average="macro",   zero_division=0),
    }

    print(f"\n=== Промпт {label} (n={metrics['n']}) ===")
    print(f"  accuracy   : {metrics['accuracy']:.4f}")
    print(f"  precision  : {metrics['precision']:.4f}")
    print(f"  recall     : {metrics['recall']:.4f}")
    print(f"  f1 (binary): {metrics['f1_binary']:.4f}")
    print(f"  f1 (macro) : {metrics['f1_macro']:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"\n  Матрица ошибок [строки=truth, столбцы=pred]:")
    print(pd.DataFrame(
        cm,
        index=["truth_legal", "truth_illegal"],
        columns=["pred_legal", "pred_illegal"],
    ))

    print(f"\n  classification_report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["legal", "illegal"],
        zero_division=0,
    ))

    return metrics


In [48]:
# ─────────────────────────────────────────────
# ГЛАВНАЯ ФУНКЦИЯ
# ─────────────────────────────────────────────
 # (ИСПОЛЬЗУЕТ ТЕСТОВЫЙ ДАТАСЕТ)
async def run_variant(client, variant_name: str, items: list):
    """
    Классифицирует все items одним вариантом промпта.
    """
    logger.info(f"{'='*50}")
    logger.info(f"Запускаем вариант: {variant_name}")
    logger.info(f"Записей: {len(items)}")

    builder = PROMPT_VARIANTS[variant_name]

    # Подготавливаем задачи с информацией о few-shot примерах
    tasks = []
    for item in items:
        messages = builder(item)
        # Извлекаем информацию о few-shot примерах из сообщений
        few_shot_ids = []
        n_examples = 0

        # Парсим few-shot примеры из user content
        user_content = messages[1]["content"] if len(messages) > 1 else ""
        example_matches = re.findall(r'ПРИМЕР ВХОДА:\s*\{[^}]*"text":\s*"([^"]*)"', user_content)
        n_examples = len(example_matches)

        # Извлекаем id примеров (если есть)
        for match in example_matches:
            # Ищем соответствующий пример в train
            for idx, text in enumerate(examples_text):
                if match[:50] in text:
                    few_shot_ids.append(examples_id[idx])
                    break

        tasks.append(
            asyncio.create_task(
                process_with_semaphore(
                    client,
                    MODEL_NAME,
                    messages,
                    item_id=item["id"],
                    few_shot_ids=few_shot_ids,
                    n_examples=n_examples,
                )
            )
        )

    start       = time.time()
    raw_results = await tqdm.gather(*tasks, desc=variant_name)
    total_time  = time.time() - start

    valid = [r for r in raw_results if not isinstance(r, Exception)]
    stats = None

    if valid:
        sum_prompt     = sum(r["prompt_tokens"]     for r in valid)
        sum_completion = sum(r["completion_tokens"] for r in valid)
        sum_total      = sum(r["total_tokens"]      for r in valid)
        avg_prompt     = sum_prompt     / len(valid)
        avg_completion = sum_completion / len(valid)
        avg_total      = sum_total      / len(valid)
        avg_time       = sum(r["time"]  for r in valid) / len(valid)

        token_summary = (
            f"[{variant_name}] СВОДКА ТОКЕНОВ | "
            f"запросов={len(valid)}/{len(items)} | "
            f"prompt: sum={sum_prompt} avg={avg_prompt:.1f} | "
            f"completion: sum={sum_completion} avg={avg_completion:.1f} | "
            f"total: sum={sum_total} avg={avg_total:.1f} | "
            f"avg_time={avg_time:.2f}s | "
            f"total_time={total_time:.2f}s"
        )
        api_logger.info(token_summary)
        logger.info(token_summary)

        print(f"\n--- Сводка токенов [{variant_name}] ---")
        print(f"  Успешных запросов : {len(valid)} / {len(items)}")
        print(f"  prompt_tokens     : sum={sum_prompt}  avg={avg_prompt:.1f}")
        print(f"  completion_tokens : sum={sum_completion}  avg={avg_completion:.1f}")
        print(f"  total_tokens      : sum={sum_total}  avg={avg_total:.1f}")
        print(f"  Среднее время/запрос : {avg_time:.2f} сек")
        print(f"  Общее время          : {total_time:.2f} сек")

        stats = {
            "time": {
                "total": total_time,
                "avg":   avg_time,
                "min":   min(r["time"] for r in valid),
                "max":   max(r["time"] for r in valid),
            },
            "prompt": {
                "sum": sum_prompt,     "avg": avg_prompt,
                "min": min(r["prompt_tokens"]     for r in valid),
                "max": max(r["prompt_tokens"]     for r in valid),
            },
            "completion": {
                "sum": sum_completion, "avg": avg_completion,
                "min": min(r["completion_tokens"] for r in valid),
                "max": max(r["completion_tokens"] for r in valid),
            },
            "total": {
                "sum": sum_total,      "avg": avg_total,
                "min": min(r["total_tokens"]      for r in valid),
                "max": max(r["total_tokens"]      for r in valid),
            },
        }
    else:
        logger.warning(f"[{variant_name}] Нет успешных ответов!")

    # Парсим ответы
    parsed = []
    for item, raw in zip(items, raw_results):
        if isinstance(raw, Exception):
            logger.warning(f"  ⚠️  Ошибка запроса для id={item['id']}: {raw}")
            continue
        result = parse_response(raw["response"], item["id"])
        if result:
            parsed.append(result)

    output_file = f"results_{variant_name}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed, f, ensure_ascii=False, indent=2)
    logger.info(f"Сохранено {len(parsed)} результатов → {output_file}")

    return parsed, stats


async def main():
    global semaphore
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    client = AsyncOpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
    )
    logger.info(f"Модель: {MODEL_NAME}")

    # Подготавливаем тестовые данные
    test_input_jsons = [row_to_input_json(row, idx) for idx, row in df_test.iterrows()]
    logger.info(f"Записей для классификации (test): {len(test_input_jsons)}")
    logger.info(f"Всего вариантов: {len(PROMPT_VARIANTS)}")

    # ── Шаг 1: классификация на тестовой выборке ─────────────────
    logger.info("Шаг 1: классификация всеми вариантами промптов на test.parquet")

    all_stats: dict[str, dict] = {}

    for variant_name in PROMPT_VARIANTS:
        parsed, stats = await run_variant(client, variant_name, test_input_jsons)
        if stats is not None:
            all_stats[variant_name] = stats

    # ── Шаг 2: метрики на тестовой выборке ───────────────────────
    logger.info("Шаг 2: расчёт метрик на тестовой выборке")
    print(f"\n{'='*60}")
    print("МЕТРИКИ НА ТЕСТОВОЙ ВЫБОРКЕ (test.parquet)")
    print(f"{'='*60}")

    all_metrics = []
    for variant_name in PROMPT_VARIANTS:
        m = evaluate_results(
            results_file=f"results_{variant_name}.json",
            df_truth=df_test,
            label=variant_name,
            truth_label_col="message_label",
        )
        if m is not None:
            all_metrics.append(m)

    if all_metrics:
        df_metrics = (
            pd.DataFrame(all_metrics)
            .set_index("version")
            .round(4)
        )

        print("\n=== Все стратегии (сортировка по f1_macro) ===")
        print(df_metrics.sort_values("f1_macro", ascending=False))

        print("\n=== Семантические стратегии ===")
        sem_mask = df_metrics.index.str.contains("semantic")
        print(df_metrics[sem_mask].sort_values("f1_macro", ascending=False))

        print("\n=== Базовые стратегии ===")
        print(df_metrics[~sem_mask].sort_values("f1_macro", ascending=False))

        metrics_file = f"metrics_test_{RUN_ID}.csv"
        df_metrics.to_csv(metrics_file)
        logger.info(f"Метрики сохранены → {metrics_file}")

        best_variant = df_metrics["f1_macro"].idxmax()
        best_f1      = df_metrics.loc[best_variant, "f1_macro"]
        logger.info(
            f"Лучший вариант по f1_macro: {best_variant} "
            f"(f1_macro={best_f1:.4f})"
        )
        print(
            f"\n🏆 Лучший вариант по f1_macro: "
            f"{best_variant} (f1_macro={best_f1:.4f})"
        )

    # ── Шаг 3: сводный df по времени и токенам ────────────────────────
    if all_stats:
        rows = []
        for variant_name, s in all_stats.items():
            rows.append({
                "variant":          variant_name,
                "time_total_s":     round(s["time"]["total"], 2),
                "time_avg_s":       round(s["time"]["avg"],   2),
                "time_min_s":       round(s["time"]["min"],   2),
                "time_max_s":       round(s["time"]["max"],   2),
                "prompt_sum":       s["prompt"]["sum"],
                "prompt_avg":       round(s["prompt"]["avg"], 1),
                "prompt_min":       s["prompt"]["min"],
                "prompt_max":       s["prompt"]["max"],
                "completion_sum":   s["completion"]["sum"],
                "completion_avg":   round(s["completion"]["avg"], 1),
                "completion_min":   s["completion"]["min"],
                "completion_max":   s["completion"]["max"],
                "total_tokens_sum": s["total"]["sum"],
                "total_tokens_avg": round(s["total"]["avg"], 1),
                "total_tokens_min": s["total"]["min"],
                "total_tokens_max": s["total"]["max"],
            })

        df_time = pd.DataFrame(rows).set_index("variant")

        print(f"\n{'='*60}")
        print("СВОДКА ПО ВРЕМЕНИ И ТОКЕНАМ")
        print(f"{'='*60}")

        print("\n--- Время выполнения (сек) ---")
        print(df_time[["time_total_s", "time_avg_s", "time_min_s", "time_max_s"]])

        print("\n--- Токены: prompt ---")
        print(df_time[["prompt_sum", "prompt_avg", "prompt_min", "prompt_max"]])

        print("\n--- Токены: completion ---")
        print(df_time[[
            "completion_sum", "completion_avg",
            "completion_min", "completion_max",
        ]])

        print("\n--- Токены: total ---")
        print(df_time[[
            "total_tokens_sum", "total_tokens_avg",
            "total_tokens_min", "total_tokens_max",
        ]])

        time_file = f"time_tokens_{RUN_ID}.csv"
        df_time.to_csv(time_file)
        logger.info(f"Сводка по времени и токенам сохранена → {time_file}")
        print(f"\n💾 Сохранено → {time_file}")

    logger.info("=" * 60)
    logger.info("ФИНАЛЬНАЯ СВОДКА — см. строки 'ТОКЕНЫ' в api_*.log")
    logger.info("=" * 60)


if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
df_train_with_embeddings.head()
df_train_with_embeddings.to_csv("df_train_with_embeddings.csv", index=False)

In [ ]:
print(df_test_with_embeddings.head())
df_test_with_embeddings.to_csv("df_test_with_embeddings.csv", index=False)

In [ ]:
print(df_val_with_embeddings.head())
df_val_with_embeddings.to_csv("df_val_with_embeddings.csv", index=False)

In [51]:
import pandas as pd

data = {
    "version": [
        "prompt_d_semantic_random_wo_dict_2exmpls",
        "prompt_d_semantic_balanced_wo_dict_2exmpls",
        "prompt_d_semantic_balanced_wo_dict_4exmpls",
        "prompt_d_semantic_random_with_dict_2exmpls",
        "prompt_d_semantic_balanced_with_dict_2exmpls"
    ],
    #"n": [305, 305, 305, 305, 305],
    "accuracy": [0.9082, 0.5049, 0.5213, 0.9148, 0.6426],
    #"precision": [0.8571, 0.4055, 0.4093, 0.8738, 0.4850],
    "recall": [0.8738, 1.0000, 0.9417, 0.8738, 0.9417],
    #"f1_binary": [0.8654, 0.5770, 0.5706, 0.8738, 0.6403],
    "f1_macro": [0.8979, 0.4901, 0.5149, 0.9047, 0.6426]
}

df = pd.DataFrame(data)
print(df)


                                        version  accuracy  recall  f1_macro
0      prompt_d_semantic_random_wo_dict_2exmpls    0.9082  0.8738    0.8979
1    prompt_d_semantic_balanced_wo_dict_2exmpls    0.5049  1.0000    0.4901
2    prompt_d_semantic_balanced_wo_dict_4exmpls    0.5213  0.9417    0.5149
3    prompt_d_semantic_random_with_dict_2exmpls    0.9148  0.8738    0.9047
4  prompt_d_semantic_balanced_with_dict_2exmpls    0.6426  0.9417    0.6426
